# プチ探究 出発点：月のデータで「基地の場所」を提案する

これは **最小の出発点** です。ここから先は自由に進めてください。

- 課題の説明：`docs/petit_inquiry_brief.md`
- ヘルパーの使い方：`docs/petit_inquiry_helpersheet.pdf`（別紙チートシート）
- コードを書かずに探索：`notebooks/explore.ipynb`（自由探索ツール）

**進め方の芯**：問いを立てる → 予想を書く → データで確かめる → 予想と違ったらなぜか考える。

In [ ]:
from moonkit import *

# データセット。中身と列を確認する
for key in DATASETS:
    df = load(key)
    print(f'■ {key}  （{len(df):,} 行）')
    print('   列:', list(df.columns))
    print()

## 例1：温度の1日の変化を見てみる

（これは「手本」です。同じことをする必要はありません。）

In [ ]:
温度 = load('温度')
赤道 = region(温度, lat=(-5, 5))
diurnal_curve(赤道)                 # 現地時間ごとの平均温度カーブ
daily_swing(赤道)[['t_mean_K', 't_swing_K', 't_std_K']].describe()

## 例2：月ぜんたいの環境を見て、地域タイプを比べる

`load('環境')` は月ぜんたいを1度マスに区切った環境指標（`temp_amp_K` 日較差、`night_min_K` 夜の底、
`noon_sun_elev_deg` 太陽高度、`earth_elev_deg` 地球の仰角＝正で表側・負で裏側）。
`region_type()` で候補地域を切り出せる。

In [ ]:
env = load('環境')
scatter(env, 'lon', 'lat', color='earth_elev_deg')   # 正＝表側 / 負＝裏側

for name in load('地域')['name']:
    r = region_type(env, name)
    print('{:32s} 日較差 {:3.0f}K  夜の底 {:3.0f}K  太陽高度 {:2.0f}°  地球の仰角 {:+3.0f}°'.format(
        name, r['temp_amp_K'].mean(), r['night_min_K'].mean(),
        r['noon_sun_elev_deg'].mean(), r['earth_elev_deg'].mean()))

---
## ここから：あなたたちの問い（RQ）

RQ は次の形にする（下のセルにコピーして埋める）：

> **〈目的〉の基地に良い場所を、〈データ〉の〈条件〉で探す。**

例：
- 「**電波天文台**の基地に良い場所を、`環境` の `earth_elev_deg`（地球が見えない裏側）と
  `temp_amp_K`（日較差が小さい）で探す」
- 「**太陽光発電**の基地に良い場所を、`環境` の `noon_sun_elev_deg`（太陽が高い）と
  `night_min_K`（夜の冷えがゆるい）で探し、赤道と南極を比べる」
- 「**長期滞在**の拠点に良い場所を、`縦孔`（溶岩チューブの天窓）の近くで探す」
- 「**氷採掘**の基地に良い場所を、`極域日照` の永久影までの距離と傾斜で探す」（南極のみ）

1. どんな基地をつくりたい？（目的）
2. その目的にとって「良い場所」とは、どのデータのどの値がどうなっている場所？
3. 予想：一番良い場所は月のどのあたり？　どの半球？　なぜ？（**分析の前に書く**）
4. 下のセルから、自由に分析を始める。困ったらチートシートと explore.ipynb。

### 問いの種（迷ったら。正解ではない）

下は使い方の例。自分の問いに関係するものだけ動かしてみる。

In [ ]:
# (a) 実在の着陸地点の環境（Apollo 11 / Chandrayaan-3 / Chang'e 4）
for lat, lon, name in [(0.67, 23.47, 'Apollo 11'), (-69.37, 32.35, 'Chandrayaan-3'), (-45.44, 177.60, "Chang'e 4")]:
    e = nearest(load('環境'), lat, lon)
    print(f"{name:14s} 日較差 {e['temp_amp_K']:.0f}K  地球の仰角 {e['earth_elev_deg']:+.0f}°  区分 {e['区分']}")

# (b) 電波天文台：裏側の候補域の中で、地球がいちばん深く隠れて温度も安定した場所
裏 = region_type(load('環境'), '裏側・赤道（電波天文の候補域）')
print(site_score(裏, {'earth_elev_deg': ('低い', 3), 'temp_amp_K': ('低い', 2)}, top=3)[
    ['lat', 'lon', 'earth_elev_deg', 'temp_amp_K']].round(1).to_string(index=False))

# (c) 長期滞在：溶岩チューブの天窓と、その場所の環境
for _, p in load('縦孔').iterrows():
    e = nearest(load('環境'), p['lat'], p['lon'])
    print(f"{p['name']:26s} 区分 {e['区分']}  日較差 {e['temp_amp_K']:.0f}K")

# (d) 氷採掘（南極のみ）：日照が高く かつ 傾斜が低い地点
pole = south_pole(load('極域日照'))
both = pole[(pole['average_illumination_percent'] >= 30) & (pole['slope_deg'] <= 8)]
print('南極 日照30%以上 かつ 傾斜8度以下:', len(both), '地点 /', len(pole))

In [ ]:
# ここから自由に。困ったらチートシートと explore.ipynb。